# conversationalchat bot 

In [78]:
import os 
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import  ChatGroq

groq_api_key = os.getenv("GROQ_API_KEY")
llm = ChatGroq(model="Llama3-8b-8192", groq_api_key=groq_api_key)
llm

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000252C9C75F10>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000252C9C79CA0>, model_name='Llama3-8b-8192', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [79]:
from langchain_chroma import Chroma
from langchain_community.document_loaders import WebBaseLoader
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.embeddings import OllamaEmbeddings

embeddings1 = (
    OllamaEmbeddings(model = "nomic-embed-text") # default for llama2 embeddings 
)

In [80]:
# second way 
os.environ["HF_TOKEN"] = os.getenv("HF_TOKEN")
from langchain_huggingface import HuggingFaceEmbeddings
embedding2 = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

In [81]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain

In [82]:
import bs4
loader = WebBaseLoader(
    web_path="https://medium.com/@nikitasinghiitk/how-to-prepare-for-quant-roles-a-complete-6-month-roadmap-1981c5050a3f",
    bs_kwargs= dict(
        parse_only= bs4.SoupStrainer(
            class_=("grecaptcha-badge", "a b c", "d e f g h i j k")
        )
    ),
)

In [83]:
docs = loader.load()
docs

[Document(metadata={'source': 'https://medium.com/@nikitasinghiitk/how-to-prepare-for-quant-roles-a-complete-6-month-roadmap-1981c5050a3f'}, page_content='Open in appSign upSign inWriteSign upSign inHomeLibraryStoriesStatsHow to prepare for Quant roles? A complete 6-month RoadmapNikita Singh·Follow6 min read·Dec 16, 2023--5ListenShareIn response to the many inquiries I’ve received about preparing for quant roles, I’ve put together a clear roadmap that covers a broad spectrum of roles in quantitative finance. It is useful for both the buy side (HFTs, hedge funds and prop shops) and sell side (investment banks) Quant roles, from pricing derivatives to mastering risk management. Whether you aspire to be a quant trader, researcher, or analyst, this roadmap is designed to provide a solid preparation plan for all these roles.Quant Toolkit (PreRequisites) :Before diving into this six-month journey, make sure you have a solid foundation in basic math (think undergraduate-level Maths 101 course

In [84]:
text_splitter = chunks = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)

splits = text_splitter.split_documents(docs)
splits

[Document(metadata={'source': 'https://medium.com/@nikitasinghiitk/how-to-prepare-for-quant-roles-a-complete-6-month-roadmap-1981c5050a3f'}, page_content='Open in appSign upSign inWriteSign upSign inHomeLibraryStoriesStatsHow to prepare for Quant roles? A complete 6-month RoadmapNikita Singh·Follow6 min read·Dec 16, 2023--5ListenShareIn response to the many inquiries I’ve received about preparing for quant roles, I’ve put together a clear roadmap that covers a broad spectrum of roles in quantitative finance. It is useful for both the buy side (HFTs, hedge funds and prop shops) and sell side (investment banks) Quant roles, from pricing derivatives to mastering risk management. Whether you aspire to be a quant trader, researcher, or analyst, this roadmap is designed to provide a solid preparation plan for all these roles.Quant Toolkit (PreRequisites) :Before diving into this six-month journey, make sure you have a solid foundation in basic math (think undergraduate-level Maths 101 course

In [85]:
vectorstore = Chroma.from_documents(documents=splits , embedding=embedding2) # can use embedding 1 too
vectorstore

In [86]:
retriever = vectorstore.as_retriever()
retriever

VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000252C9BCA660>, search_kwargs={})

In [87]:
# prompt template 
system_prompt = (
    "you are a assistant for question answer tasks"
    "use the following recieved context data to answer"
    "if you dont know the answer say that you don't know"
    "keep the answer consise straight to the point"
    "\n\n"
    "{context}"
)

prompt = ChatPromptTemplate.from_messages(
    [
        ("system",system_prompt),
        ("human","{input}")
    ]
)

In [88]:
question_answer_chain = create_stuff_documents_chain(llm , prompt)# give data to llm in prompt style 
rag_chain = create_retrieval_chain(retriever , question_answer_chain)  # main chain using retriever of db data 
rag_chain

RunnableBinding(bound=RunnableAssign(mapper={
  context: RunnableBinding(bound=RunnableLambda(lambda x: x['input'])
           | VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000252C9BCA660>, search_kwargs={}), kwargs={}, config={'run_name': 'retrieve_documents'}, config_factories=[])
})
| RunnableAssign(mapper={
    answer: RunnableBinding(bound=RunnableBinding(bound=RunnableAssign(mapper={
              context: RunnableLambda(format_docs)
            }), kwargs={}, config={'run_name': 'format_inputs'}, config_factories=[])
            | ChatPromptTemplate(input_variables=['context', 'input'], input_types={}, partial_variables={}, messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=['context'], input_types={}, partial_variables={}, template="you are a assistant for question answer tasksuse the following recieved context data to answerif you dont know the answer say that you don't 

In [89]:
response = rag_chain.invoke(
    {"input" : "howe to get into Quant roles ?"}
)

response['answer']

'According to the article, to get into Quant roles, you can follow a 6-month roadmap that covers a broad spectrum of roles in quantitative finance.'

# adding chat history

In [90]:
from langchain.chains import create_history_aware_retriever
from langchain_core.prompts import MessagesPlaceholder


contextualize_q_system_prompt = (
    "given chat history and latest user question"
    "which might reference context in chat history"
    "formulate a standalone question which can be understood"
    "without the chat history , do not answer the question"
    "just reformulate if needed and otherwise return as it is."
)

contextualize_q_prompt = ChatPromptTemplate.from_messages(
    [
        ("system" , contextualize_q_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human","{input}")
    ]
)

history_aware_retriever = create_history_aware_retriever(llm , retriever , contextualize_q_prompt)
history_aware_retriever

RunnableBinding(bound=RunnableBranch(branches=[(RunnableLambda(lambda x: not x.get('chat_history', False)), RunnableLambda(lambda x: x['input'])
| VectorStoreRetriever(tags=['Chroma', 'HuggingFaceEmbeddings'], vectorstore=<langchain_chroma.vectorstores.Chroma object at 0x00000252C9BCA660>, search_kwargs={}))], default=ChatPromptTemplate(input_variables=['chat_history', 'input'], input_types={'chat_history': list[typing.Annotated[typing.Union[typing.Annotated[langchain_core.messages.ai.AIMessage, Tag(tag='ai')], typing.Annotated[langchain_core.messages.human.HumanMessage, Tag(tag='human')], typing.Annotated[langchain_core.messages.chat.ChatMessage, Tag(tag='chat')], typing.Annotated[langchain_core.messages.system.SystemMessage, Tag(tag='system')], typing.Annotated[langchain_core.messages.function.FunctionMessage, Tag(tag='function')], typing.Annotated[langchain_core.messages.tool.ToolMessage, Tag(tag='tool')], typing.Annotated[langchain_core.messages.ai.AIMessageChunk, Tag(tag='AIMessag

In [91]:
qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system" , system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human","{input}")
    ]
)

In [92]:
question_answer_chain = create_stuff_documents_chain(llm,qa_prompt)
rag_chain2 = create_retrieval_chain(history_aware_retriever , question_answer_chain)

In [93]:
from langchain_core.messages import AIMessage , HumanMessage 

chat_history = []
question = "what is quant finance?"
response1 = rag_chain.invoke(
    {"input": question , "chat_history": chat_history}
)

chat_history.extend(
    [
        HumanMessage(content=question),
        AIMessage(content = response1["answer"])
    ]
)

question2 = "what is the way to get a job  in it  ? tell me about the skills i need"
response2 = rag_chain2.invoke(
    {"input":question2 , "chat_history":chat_history}
)

print(response2["answer"])

To get a job in quantitative finance, you typically need to have a strong foundation in:

1. Mathematics: Linear Algebra, Calculus, Probability, Statistics, and Optimization.
2. Computer Science: Programming skills in languages such as Python, C++, Java, and R.
3. Finance: Understanding of financial markets, instruments, and concepts such as risk management, derivatives, and portfolio optimization.

For a quant finance role, you can expect the following skills:

1. Programming languages: Python is often preferred, but knowledge of languages like C++, Java, and R is also valuable.
2. Data analysis and visualization: Familiarity with libraries like Pandas, NumPy, and Matplotlib in Python, or similar tools in other languages.
3. Mathematical modeling: Knowledge of mathematical techniques such as linear algebra, calculus, probability, and optimization.
4. Data science: Understanding of data science concepts like data preprocessing, machine learning, and statistical modeling.
5. Financial m

In [94]:
### message history 
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory

store = {}

def get_session_history(session_id : str)->BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain2,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",
    output_messages_key="answer",
)


### 
conversational_rag_chain.invoke(
    {"input" : "what are the skills required for quantitative finance and how much time to achieve it?"},
    config={
        "configurable":{"session_id":"abc123"}
    } # construct as key "abc123" in store 
)["answer"]


"Based on the provided roadmap, the skills required for quantitative finance include:\n\n1. Undergraduate-level Maths (101 courses)\n2. Essential computer science concepts (like CS 101)\n3. Quantitative finance skills, including:\n\t* Bond pricing\n\t* Time value of money concepts\n\t* Present value calculations\n\t* Fixed income concepts\n\nAs for the time required to achieve these skills, it depends on the individual's prior knowledge and experience. However, here's a rough estimate of the time it may take to achieve each skill:\n\n1. Undergraduate-level Maths (101 courses): 2-3 months\n2. Essential computer science concepts (like CS 101): 2-3 months\n3. Quantitative finance skills:\n\t* Bond pricing: 1-2 months\n\t* Time value of money concepts: 1-2 weeks\n\t* Present value calculations: 1-2 weeks\n\t* Fixed income concepts: 2-3 months\n\nAssuming you start from scratch, it may take around 6-12 months to achieve a basic understanding of the required skills. However, this is just a r

In [95]:
conversational_rag_chain.invoke(
    {"input" : "what computer science skills are required ?"},
    config={
        "configurable":{"session_id":"abc124"}
    } # construct as key "abc123" in store 
)["answer"]

"Based on the provided context, the following computer science skills are required:\n\n* Algorithms: basic algorithms and their efficiency, including:\n\t+ Sorting algorithms (quick sort, merge sort)\n\t+ Search algorithms (binary search, linear search)\n* Data Structures: understanding the importance of data structures in quantitative analysis and implementing them in financial algorithms, including:\n\t+ Arrays\n\t+ Linked lists\n\t+ Graphs and trees\n\t+ Hash tables and their applications\n* Time and space complexity analysis\n* Numerical Methods: solving mathematical problems numerically, applying these methods in finance and modeling, including:\n\t+ Root-finding methods (Newton's method, bisection method)\n\t+ Numerical integration and differentiation\n\nThese skills are likely to be required for a Quantitative Finance role."

ValidationError: 1 validation error for ChatGroq
model
  Field required [type=missing, input_value={'api_key': 'gsk_ITCStMDd...O9', 'model_kwargs': {}}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/missing

ChatGroq(client=<groq.resources.chat.completions.Completions object at 0x00000252CF217380>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000252CF217FE0>, model_name='Llama3-8b-8192', model_kwargs={}, groq_api_key=SecretStr('**********'))

llama-guard-3-8b
qwen-2.5-32b
deepseek-r1-distill-llama-70b
llama-3.1-8b-instant
mistral-saba-24b
llama-3.2-1b-preview
llama3-70b-8192
qwen-2.5-coder-32b
llama-3.3-70b-specdec
gemma2-9b-it
deepseek-r1-distill-qwen-32b
allam-2-7b
qwen-qwq-32b
llama3-8b-8192
llama-3.2-11b-vision-preview
llama-3.3-70b-versatile
distil-whisper-large-v3-en
whisper-large-v3
llama-3.2-3b-preview
llama-3.2-90b-vision-preview
whisper-large-v3-turbo
